# Подключим Spark

In [ ]:
!pip install pyspark==3.4.1
from pyspark.sql import SparkSession
import sys

Инициализируем ключи доступа к S3

In [ ]:
from pyspark.sql import SparkSession

ACCESS_KEY = "lab-1-user-7"
SECRET_KEY = "FIq9Q160hYaeYjvbAvAv"
ENDPOINT = "http://minio-1.devops-playground.innopolis.university"
BUCKET = "lab-1-bucket-1"
YOUR_NAME = "kirill-gusev"

# Создание SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("Spark Lab") \
    .master("local[*]") \
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .getOrCreate()

sc = spark.sparkContext
print("SparkSession создана")

SparkSession создана


# Parallelize и coalesce

In [ ]:
# Создание RDD
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
rdd = sc.parallelize(data, 5)
print(f"Исходное количество партиций: {rdd.getNumPartitions()}")

# Сохранение с 5 партициями
!rm -rf /content/rdd_output_5parts
rdd.saveAsTextFile("/content/rdd_output_5parts")
print("✅ Сохранено с 5 партициями")

# Уменьшение партиций
rdd_coalesced = rdd.coalesce(2)
print(f"После coalesce: {rdd_coalesced.getNumPartitions()}")

# Сохранение с 2 партициями
!rm -rf /content/rdd_output_2parts
rdd_coalesced.saveAsTextFile("/content/rdd_output_2parts")
print("✅ Сохранено с 2 партициями")

Исходное количество партиций: 5
✅ Сохранено с 5 партициями
После coalesce: 2
✅ Сохранено с 2 партициями


# Загрузим нужные библитеки для чтения файлов с S3 + подключимся к хранилищу

In [ ]:
!pip install boto3 pandas

In [ ]:
import boto3

In [ ]:
import pandas as pd

In [ ]:
from io import StringIO, BytesIO

Добавим модуль `boto3` для исправления технической ошибки подключения к S3 (после подключения к Minio конвертируем в `spark.createDataFrame`)

Настроим подключение к S3

In [ ]:
endpoint = 'https://minio-1.devops-playground.innopolis.university'
access_key = 'lab-1-user-7'
secret_key = 'FIq9Q160hYaeYjvbAvAv'

Прочитаем файл

In [ ]:
bucket_name = 'lab-1-bucket-1'
object_key = 'kirill-gusev/retail/Online_Retail-1.csv'

In [ ]:
session = boto3.session.Session()

s3_client = session.client(

    service_name='s3',
    endpoint_url='https://minio-1.devops-playground.innopolis.university',
    aws_access_key_id='lab-1-user-7',
    aws_secret_access_key='FIq9Q160hYaeYjvbAvAv',
    verify=False

)

In [ ]:
response = s3_client.get_object(Bucket='lab-1-bucket-1', Key='kirill-gusev/retail/Online_Retail-1.csv')
content = response['Body'].read().decode('utf-8')
df_pd = pd.read_csv(StringIO(content))

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Конвертируем в `spark.createDataFrame`

In [ ]:
df_spark = spark.createDataFrame(df_pd)

In [ ]:
df_spark.show(8)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS S

# Произведем объединение 2-х файлов

Прочитаем 2-й файл

In [ ]:
response2 = s3_client.get_object(

    Bucket='lab-1-bucket-1',
    Key='kirill-gusev/retail/Online_Retail-2.csv'  # Обратите внимание: Online, а не 0nline!

)

content2 = response2['Body'].read().decode('utf-8')
df_pd2 = pd.read_csv(StringIO(content2))

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Конвертируем в Spark

In [ ]:
df_spark2 = spark.createDataFrame(df_pd2)

Произведем объединение

In [ ]:
df_union = df_spark.union(df_spark2)
print(f"Всего строк: {df_union.count()}")
df_union.show(5)

Всего строк: 541909
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only sho

# Сохраним данные в Parquet

Для начало сохраним данные локально

In [ ]:
local_path = "/content/retail_parquet"
df_union.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(local_path) ;     print(f"Данные сохранены локально: {local_path}")

Данные сохранены локально: /content/retail_parquet


Теперь произведем загрузку данных в S3

In [ ]:
import os ; from botocore.client import Config

In [ ]:
for root, dirs, files in os.walk(local_path):
    for file in files:
        local_file = os.path.join(root, file)
        s3_key = f"kirill-gusev/retail_parquet/{file}"

        with open(local_file, 'rb') as f:
            s3_client.put_object(
                Bucket=BUCKET,
                Key=s3_key,
                Body=f
            )
        print(f"Загружен: {s3_key}")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Загружен: kirill-gusev/retail_parquet/part-00002-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Загружен: kirill-gusev/retail_parquet/.part-00000-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Загружен: kirill-gusev/retail_parquet/_SUCCESS


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Загружен: kirill-gusev/retail_parquet/.part-00001-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Загружен: kirill-gusev/retail_parquet/part-00000-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet


# Добавляем колонку InvoiceMonth

Подгрузим дополнительные модули и библиотеки

In [ ]:
import pyspark.sql.functions as F
import pandas as pd
from io import BytesIO
import pyarrow.parquet as pq
import pyarrow as pa

Получим список всех parquet файлов в папке



In [ ]:
response = s3_client.list_objects_v2(

    Bucket=BUCKET,
    Prefix=f"{YOUR_NAME}/retail_parquet/"
)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Проверим содержимое

In [ ]:
response = s3_client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=f"{YOUR_NAME}/retail_parquet/"
)

if 'Contents' in response:
    for obj in response['Contents']:
        print(f"  - {obj['Key']} ({obj['Size']} bytes)")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  - kirill-gusev/retail_parquet/._SUCCESS.crc (8 bytes)
  - kirill-gusev/retail_parquet/.part-00000-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc (6012 bytes)
  - kirill-gusev/retail_parquet/.part-00000-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc (6012 bytes)
  - kirill-gusev/retail_parquet/.part-00001-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc (6284 bytes)
  - kirill-gusev/retail_parquet/.part-00001-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc (6284 bytes)
  - kirill-gusev/retail_parquet/.part-00002-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc (8172 bytes)
  - kirill-gusev/retail_parquet/.part-00002-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc (8172 bytes)
  - kirill-gusev/retail_parquet/.part-00003-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc (7888 bytes)
  - kirill-gusev/retail_parquet/.part-00003-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc (7888 bytes)


Скрипт получения и обьединения файлов parquet

In [ ]:
response = s3_client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=f"{YOUR_NAME}/retail_parquet/"
)

all_data = []

if 'Contents' in response:
    for obj in response['Contents']:
        # Проверяем, что это parquet файл (по расширению)
        if obj['Key'].endswith('.parquet'):
            print(f"Чтение: {obj['Key']}")

            # Скачиваем файл
            file_response = s3_client.get_object(Bucket=BUCKET, Key=obj['Key'])

            # Читаем parquet
            df_pd = pd.read_parquet(BytesIO(file_response['Body'].read()))

            print(f"{len(df_pd)} строк")
            all_data.append(df_pd)
        else:
            print(f"Пропускаем (не parquet): {obj['Key']}")
else:
    print("Папка не найдена или пуста")

# Проверяем, что нашли файлы
print(f"\nНайдено parquet файлов: {len(all_data)}")

if len(all_data) > 0:
    df_pd_all = pd.concat(all_data, ignore_index=True)
    print(f"Всего строк в объединенном файле: {len(df_pd_all)}")
    print("\nПервые 5 строк:")
    print(df_pd_all.head())
else:
    print("Нет parquet файлов для обработки")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Пропускаем (не parquet): kirill-gusev/retail_parquet/._SUCCESS.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00000-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00000-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00001-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00001-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00002-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00002-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parquet/.part-00003-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet.crc
Пропускаем (не parquet): kirill-gusev/retail_parque

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   → 114688 строк
📖 Чтение: kirill-gusev/retail_parquet/part-00000-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

   → 156672 строк
📖 Чтение: kirill-gusev/retail_parquet/part-00003-3aa0d93b-8cb9-4920-ae6a-6f3cfc0fe55f-c000.snappy.parquet


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   → 155731 строк
📖 Чтение: kirill-gusev/retail_parquet/part-00003-7caeaa95-fa4f-4ad5-aa28-178c0ed53b90-c000.snappy.parquet


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   → 155731 строк

Найдено parquet файлов: 8
Всего строк в объединенном файле: 1083818

Первые 5 строк:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55       17850  United Kingdom  
1  2010-12-01 08:26:00       3.39       17850  United Kingdom  
2  2010-12-01 08:26:00       2.75       17850  United Kingdom  
3  2010-12-01 08:26:00       3.39       17850  United Kingdom  
4  2010-12-01 08:26:00       3.39       17850  United Kingdom  


### Добавим колонку InvoiceMonth

In [ ]:
# Преобразуем колонку InvoiceDate в datetime
df_pd_all['InvoiceDate'] = pd.to_datetime(df_pd_all['InvoiceDate'])

# Добавляем колонку с первым днем месяца
df_pd_all['InvoiceMonth'] = df_pd_all['InvoiceDate'].dt.to_period('M').dt.start_time

print("Результат - первые 10 строк с новой колонкой:")
print(df_pd_all[['InvoiceDate', 'InvoiceMonth']].head(10))

print("\nУникальные месяцы в данных:")
print(df_pd_all['InvoiceMonth'].unique())

Результат - первые 10 строк с новой колонкой:
          InvoiceDate InvoiceMonth
0 2010-12-01 08:26:00   2010-12-01
1 2010-12-01 08:26:00   2010-12-01
2 2010-12-01 08:26:00   2010-12-01
3 2010-12-01 08:26:00   2010-12-01
4 2010-12-01 08:26:00   2010-12-01
5 2010-12-01 08:26:00   2010-12-01
6 2010-12-01 08:26:00   2010-12-01
7 2010-12-01 08:28:00   2010-12-01
8 2010-12-01 08:28:00   2010-12-01
9 2010-12-01 08:34:00   2010-12-01

Уникальные месяцы в данных:
<DatetimeArray>
['2010-12-01 00:00:00', '2011-01-01 00:00:00', '2011-02-01 00:00:00',
 '2011-03-01 00:00:00', '2011-04-01 00:00:00', '2011-05-01 00:00:00',
 '2011-06-01 00:00:00', '2011-07-01 00:00:00', '2011-08-01 00:00:00',
 '2011-09-01 00:00:00', '2011-10-01 00:00:00', '2011-11-01 00:00:00',
 '2011-12-01 00:00:00']
Length: 13, dtype: datetime64[ns]


### Сохраним данные с новой колонкой

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Сохраняем в parquet
buffer = BytesIO()
df_pd_all.to_parquet(buffer, index=False, engine='pyarrow')
buffer.seek(0)

# Загружаем в S3
month_path = f"{YOUR_NAME}/retail_with_month/data.parquet"
s3_client.put_object(
    Bucket=BUCKET,
    Key=month_path,
    Body=buffer.getvalue()
)

print(f"Данные с колонкой InvoiceMonth сохранены в: {month_path}")

response = s3_client.get_object(Bucket=BUCKET, Key=month_path)
df_check = pd.read_parquet(BytesIO(response['Body'].read()))

print(f"Проверка: {len(df_check)} строк сохранено")
print(df_check[['InvoiceDate', 'InvoiceMonth']].head())

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Данные с колонкой InvoiceMonth сохранены в: kirill-gusev/retail_with_month/data.parquet


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Проверка: 1083818 строк сохранено
          InvoiceDate InvoiceMonth
0 2010-12-01 08:26:00   2010-12-01
1 2010-12-01 08:26:00   2010-12-01
2 2010-12-01 08:26:00   2010-12-01
3 2010-12-01 08:26:00   2010-12-01
4 2010-12-01 08:26:00   2010-12-01


# Запрос без партиционирования (через pandas)

In [ ]:
# Уникальные клиенты за декабрь 2010

dec_2010 = df_pd_all[df_pd_all['InvoiceMonth'] == '2010-12-01']
customers_dec = dec_2010['CustomerID'].dropna().unique()

print(f"Уникальные клиенты за декабрь 2010: {len(customers_dec)}")
print("Первые 10:")
print(customers_dec[:10])

# Группировка по клиентам (альтернативный запрос)

customer_stats = dec_2010.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'UnitPrice': 'mean'
}).rename(columns={
    'InvoiceNo': 'orders_count',
    'Quantity': 'total_items',
    'UnitPrice': 'avg_price'
}).reset_index()

print("Статистика по клиентам за декабрь 2010:")
print(customer_stats.head(10))

Уникальные клиенты за декабрь 2010: 949
Первые 10:
[17850 13047 12583 13748 15100 15291 14688 17809 15311 14527]
Статистика по клиентам за декабрь 2010:
   CustomerID  orders_count  total_items  avg_price
0       12347             1          638   2.890000
1       12348             1         2508   2.917647
2       12370             2         1936   2.894286
3       12377             1         1208   2.106279
4       12383             1         1508   1.325135
5       12386             1          428   2.726250
6       12395             2         1506   3.390000
7       12417             1          120   7.003636
8       12423             1          376   2.636875
9       12427             2          140   5.226923


# Партиционирование по месяцу

In [ ]:
import os

# Группируем данные по месяцам
df_pd_all['InvoiceMonth'] = pd.to_datetime(df_pd_all['InvoiceMonth']).dt.strftime('%Y-%m-%d')

for month, group in df_pd_all.groupby('InvoiceMonth'):
    print(f"Сохраняем данные за {month}...")

    # Сохраняем в буфер
    buffer = BytesIO()
    group.to_parquet(buffer, index=False, engine='pyarrow')
    buffer.seek(0)

    # Загружаем в S3 в соответствующую папку
    month_path = f"{YOUR_NAME}/retail_partitioned/InvoiceMonth={month}/data.parquet"
    s3_client.put_object(
        Bucket=BUCKET,
        Key=month_path,
        Body=buffer.getvalue()
    )
    print(f"{len(group)} строк сохранено в {month_path}")

print("Все данные разбиты по месяцам и сохранены")

Сохраняем данные за 2010-12-01...


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


84962 строк сохранено в kirill-gusev/retail_partitioned/InvoiceMonth=2010-12-01/data.parquet
Сохраняем данные за 2011-01-01...


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


70294 строк сохранено в kirill-gusev/retail_partitioned/InvoiceMonth=2011-01-01/data.parquet
Сохраняем данные за 2011-02-01...


...

121484 строк сохранено в kirill-gusev/retail_partitioned/InvoiceMonth=2011-10-01/data.parquet
Сохраняем данные за 2011-11-01...


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


169422 строк сохранено в kirill-gusev/retail_partitioned/InvoiceMonth=2011-11-01/data.parquet
Сохраняем данные за 2011-12-01...


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


51050 строк сохранено в kirill-gusev/retail_partitioned/InvoiceMonth=2011-12-01/data.parquet
Все данные разбиты по месяцам и сохранены


Проверим как прошло партиционирование

In [ ]:
# Проверяем структуру папок
response = s3_client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=f"{YOUR_NAME}/retail_partitioned/"
)

folders = set()
for obj in response.get('Contents', []):
    parts = obj['Key'].split('/')
    if len(parts) > 2 and 'InvoiceMonth=' in parts[1]:
        folders.add(parts[1])

print("Созданные партиции: \n")
for folder in sorted(folders):
    print(f"  - {folder}")

Созданные партиции: 



/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'minio-1.devops-playground.innopolis.university'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
